# AKATSUKI Training — Qwen2.5-7B-Instruct QLoRA

**Base model:** Qwen/Qwen2.5-7B-Instruct (Qwen2.5-3B-Instruct also supported)

Menu → Runtime → Change runtime type → **T4 GPU**

---

### Cell 1 — Hardware check

In [ ]:
import torch, psutil
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {gpu.total_mem/1e9:.1f}GB | CUDA: {torch.version.cuda}")
print(f"RAM: {psutil.virtual_memory().total/1e9:.1f}GB | CPU: {psutil.cpu_count()} cores")

### Cell 2 — Install requirements

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q transformers accelerate bitsandbytes peft trl datasets scipy
!pip install -q pyyaml jinja2
print("Done")

### Cell 3 — Clone AKATSUKI

In [ ]:
import os
if not os.path.exists("akatsuki"):
    !git clone https://github.com/kwkkd/akatsuki-hermes-integration akatsuki
%cd akatsuki

### Cell 4 — Configure (Qwen2.5-7B-Instruct)

In [ ]:
%%writefile akatsuki.yaml
model:
  id: "Qwen/Qwen2.5-7B-Instruct"
  # id: "Qwen/Qwen2.5-3B-Instruct"  # swap for faster training
  lora_path: "./hacker_ai_model"
  merged_path: "./merged_hacker_ai_model"
  use_4bit: true
  max_seq_length: 4096
training:
  num_epochs: 3
  batch_size: 1
  gradient_accumulation_steps: 8
  learning_rate: 0.0002
  lora_r: 16
  lora_alpha: 32
  dpo_beta: 0.1
api:
  host: "0.0.0.0"
  port: 8000
  max_tokens: 2048
  temperature: 0.7
telegram:
  bot_token_env: "TELEGRAM_BOT_TOKEN"
  allowed_ids_env: "TELEGRAM_ALLOWED_IDS"

### Cell 5 — Generate dataset (300 security conversation samples)

In [ ]:
from dataset_builder import build_dataset
samples = build_dataset("dataset.jsonl", num_samples=300)
print(f"Generated {len(samples)} samples")

### Cell 6 — QLoRA SFT Training

- Qwen2.5-7B-Instruct + 4-bit NF4 + LoRA r=16
- T4 (16GB): ~45 min/epoch, ~2.5h total for 3 epochs
- Qwen2.5-3B-Instruct: ~15 min/epoch

In [ ]:
%run train.py

### Cell 7 — Inference smoke test

In [ ]:
from inference import AkatsukiInference
from akatsuki_config import CONFIG

ai = AkatsukiInference(model_path=CONFIG.model.merged_path)
ai.load()

tests = [
    "너는 누구야?",
    "@Pain, C2 chain 구축해줘",
    "nmap SMB 취약점 스캔 명령어",
]
for t in tests:
    print(f"=== {t} ===")
    print(ai.chat(t))
    print()

### Cell 8 — Save to Google Drive

In [ ]:
from google.colab import drive
import shutil
drive.mount("/content/drive")
dest = "/content/drive/MyDrive/akatsuki_7b_merged"
if os.path.exists(dest):
    shutil.rmtree(dest)
shutil.copytree(CONFIG.model.merged_path, dest)
print(f"Saved to Google Drive: {dest}")